# Benchmark solver: appsi_highs vs highspy directo (LP)

Compara el flujo **actual** de la app (`SolverFactory('appsi_highs')` sobre instancia Pyomo)
con la **alternativa A**: escribir `.lp` + `highspy.readModel()` + `run()` + mapeo de vuelta.

**Datos:** CSVs del escenario regional (Caso 1), recortados a años en `BENCHMARK_YEARS`.

**Hilos HiGHS:** en la celda 1, `BENCHMARK_THREADS`:
- `0` → todos los CPUs (`os.cpu_count()`, igual que la app con `SIM_SOLVER_THREADS=0`)
- `N > 0` → exactamente N hilos (ej. `4`, `8`, `12`)

**Kernel:** `backend/.venv` (Python con pyomo, highspy, app).

In [15]:
# Celda 1 — Setup
from __future__ import annotations

import copy
import os
import shutil
import sys
import tempfile
import zipfile
from pathlib import Path
from time import perf_counter

import pandas as pd
import pyomo.environ as pyo
from IPython.display import display
from pyomo.core import Constraint, Suffix, Var

import highspy

# Raíz del repo y backend en sys.path
REPO_ROOT = Path.cwd().resolve()
if (REPO_ROOT / "backend" / "app").is_dir():
    BACKEND_ROOT = REPO_ROOT / "backend"
elif (REPO_ROOT.parent / "backend" / "app").is_dir():
    REPO_ROOT = REPO_ROOT.parent
    BACKEND_ROOT = REPO_ROOT / "backend"
else:
    raise RuntimeError(
        "Ejecuta el notebook desde la raíz del repo o desde notebooks/"
    )

if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

from app.simulation.core.data_processing import (
    eliminar_valores_fuera_de_indices,
    get_processing_result_from_csv_dir,
    normalize_mode_of_operation_in_csv_dir,
    reorder_activity_ratio_csvs_for_dataportal,
    strip_whitespace_in_set_csvs,
)
from app.simulation.core.instance_builder import build_instance
from app.simulation.core.model_definition import create_abstract_model
from app.simulation.core.solver import _effective_solver_threads

CSV_ZIP = Path(
    "/home/jchavez/Documentos/UPME/Datos Simulacion/Regional/Caso 1/CSV.zip"
)
BENCHMARK_YEARS = {2022, 2023, 2024, 2025}
WORK_DIR = Path(tempfile.mkdtemp(prefix="osemosys_benchmark_"))
CSV_DIR = WORK_DIR / "csv"
LP_PATH = WORK_DIR / "model.lp"

# --- Hilos HiGHS (misma lógica que solver.py en producción) ---
# 0  → todos los CPUs (os.cpu_count())
# N  → número fijo de hilos (ej. 4, 8, 12)
# None → leer SIM_SOLVER_THREADS del entorno
BENCHMARK_THREADS: int | None = 0
_threads_config = (
    BENCHMARK_THREADS
    if BENCHMARK_THREADS is not None
    else int(os.getenv("SIM_SOLVER_THREADS", "0") or 0)
)
SOLVER_THREADS_EFFECTIVE = _effective_solver_threads(_threads_config)
if _threads_config > 0:
    SOLVER_THREADS_LABEL = str(_threads_config)
else:
    SOLVER_THREADS_LABEL = f"all({SOLVER_THREADS_EFFECTIVE})"

print("REPO_ROOT:", REPO_ROOT)
print("BACKEND_ROOT:", BACKEND_ROOT)
print("WORK_DIR:", WORK_DIR)
print("highspy: OK")
print("appsi_highs disponible:", pyo.SolverFactory("appsi_highs").available(exception_flag=False))
print(
    f"Hilos HiGHS: config={_threads_config!r} → {SOLVER_THREADS_LABEL} "
    f"(efectivo={SOLVER_THREADS_EFFECTIVE}, cpus={os.cpu_count()})"
)

REPO_ROOT: /home/jchavez/Documentos/APPS/UPME/Osemosys_UPME
BACKEND_ROOT: /home/jchavez/Documentos/APPS/UPME/Osemosys_UPME/backend
WORK_DIR: /tmp/osemosys_benchmark_0id0kjvz
highspy: OK
appsi_highs disponible: True
Hilos HiGHS: config=0 → all(16) (efectivo=16, cpus=16)


In [16]:
# Celda 2 — Descomprimir CSVs, recortar a 4 años y construir UNA instancia Pyomo

def trim_csvs_to_years(csv_dir: Path, keep_years: set[int]) -> None:
    """Filtra todos los CSVs para conservar solo los años indicados."""
    year_csv = csv_dir / "YEAR.csv"
    if year_csv.is_file():
        df = pd.read_csv(year_csv)
        col = df.columns[0]
        df = df[df[col].astype(int).isin(keep_years)]
        df.to_csv(year_csv, index=False)

    for csv_file in csv_dir.glob("*.csv"):
        if csv_file.name == "YEAR.csv":
            continue
        df = pd.read_csv(csv_file, low_memory=False)
        if "YEAR" in df.columns:
            df["YEAR"] = pd.to_numeric(df["YEAR"], errors="coerce")
            df = df[df["YEAR"].isin(keep_years)]
            df.to_csv(csv_file, index=False)


def unzip_csvs(zip_path: Path, dest_csv_dir: Path) -> Path:
    if not zip_path.is_file():
        raise FileNotFoundError(f"No existe: {zip_path}")
    dest_csv_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dest_csv_dir.parent)
    # El zip suele traer carpeta CSV/
    nested = dest_csv_dir.parent / "CSV"
    if nested.is_dir() and not any(dest_csv_dir.iterdir()):
        for item in nested.iterdir():
            shutil.move(str(item), str(dest_csv_dir / item.name))
        nested.rmdir()
    return dest_csv_dir


def preprocess_csv_dir(csv_dir: Path) -> None:
    """Mismo preprocesamiento que run_osemosys_from_csv_dir en osemosys_core."""
    reorder_activity_ratio_csvs_for_dataportal(str(csv_dir))
    normalize_mode_of_operation_in_csv_dir(str(csv_dir))
    strip_whitespace_in_set_csvs(str(csv_dir))
    eliminar_valores_fuera_de_indices(str(csv_dir))


def build_concrete_instance(csv_dir: Path):
    preprocess_csv_dir(csv_dir)
    proc = get_processing_result_from_csv_dir(str(csv_dir))
    has_storage = proc.has_storage
    has_udc = proc.has_udc
    print(f"has_storage={has_storage}, has_udc={has_udc}")
    print(
        f"REGION={len(proc.sets.get('REGION', []))}, "
        f"TECH={len(proc.sets.get('TECHNOLOGY', []))}, "
        f"YEAR={len(proc.sets.get('YEAR', []))}, "
        f"TS={len(proc.sets.get('TIMESLICE', []))}"
    )
    t0 = perf_counter()
    abstract = create_abstract_model(has_storage=has_storage, has_udc=has_udc)
    instance = build_instance(
        abstract,
        str(csv_dir),
        has_storage=has_storage,
        has_udc=has_udc,
    )
    elapsed = perf_counter() - t0
    return instance, elapsed, has_storage, has_udc


csv_dir = unzip_csvs(CSV_ZIP, CSV_DIR)
trim_csvs_to_years(csv_dir, BENCHMARK_YEARS)
print("CSV_DIR:", csv_dir, "archivos:", len(list(csv_dir.glob("*.csv"))))
print("Años del benchmark:", sorted(BENCHMARK_YEARS))

instance, build_seconds, HAS_STORAGE, HAS_UDC = build_concrete_instance(csv_dir)
print(f"Instancia construida en {build_seconds:.2f} s (una sola copia en RAM)")

CSV_DIR: /tmp/osemosys_benchmark_0id0kjvz/csv archivos: 57
Años del benchmark: [2022, 2023, 2024, 2025]
has_storage=True, has_udc=False
REGION=1, TECH=2343, YEAR=4, TS=1
           0 seconds to construct Set YEAR; 1 index total
           0 seconds to construct Set TECHNOLOGY; 1 index total
           0 seconds to construct Set TIMESLICE; 1 index total
           0 seconds to construct Set FUEL; 1 index total
           0 seconds to construct Set EMISSION; 1 index total
           0 seconds to construct Set MODE_OF_OPERATION; 1 index total
           0 seconds to construct Set REGION; 1 index total
           0 seconds to construct Set STORAGE; 1 index total
           0 seconds to construct Set SEASON; 1 index total
           0 seconds to construct Set DAYTYPE; 1 index total
           0 seconds to construct Set DAILYTIMEBRACKET; 1 index total
           0 seconds to construct Set STORAGEINTRADAY; 1 index total
           0 seconds to construct Set STORAGEINTRAYEAR; 1 index total
   

In [17]:
# Celda 3 — Benchmark 1: appsi_highs (flujo actual)


def benchmark_appsi_highs(
    instance,
    lp_path: Path | None = None,
    *,
    threads: int = 0,
) -> dict:
    write_lp_seconds = 0.0
    lp_size_mb = 0.0
    if lp_path is not None:
        lp_path = Path(lp_path)
        lp_path.parent.mkdir(parents=True, exist_ok=True)
        t_w = perf_counter()
        instance.write(
            filename=str(lp_path),
            io_options={"symbolic_solver_labels": True},
        )
        write_lp_seconds = perf_counter() - t_w
        lp_size_mb = lp_path.stat().st_size / (1024 * 1024)

    solver = pyo.SolverFactory("appsi_highs")
    if not solver.available(exception_flag=False):
        raise RuntimeError("appsi_highs no disponible")
    opts = getattr(solver, "highs_options", None)
    if isinstance(opts, dict):
        opts["threads"] = threads
    if hasattr(solver, "config"):
        try:
            solver.config.stream_solver = False
            solver.config.load_solution = False
        except Exception:
            pass

    t0 = perf_counter()
    results = solver.solve(instance, tee=False, load_solutions=False)
    solve_seconds = perf_counter() - t0

    raw_status = str(results.solver.termination_condition)
    obj = 0.0
    if "optimal" in raw_status.lower():
        t_load = perf_counter()
        instance.solutions.load_from(results)
        load_seconds = perf_counter() - t_load
        try:
            obj = float(pyo.value(instance.OBJ))
        except Exception:
            pass
    else:
        load_seconds = 0.0

    return {
        "backend": "appsi_highs",
        "status": raw_status,
        "objective": obj,
        "write_lp_seconds": write_lp_seconds,
        "lp_size_mb": lp_size_mb,
        "solve_seconds": solve_seconds,
        "load_solution_seconds": load_seconds,
        "total_seconds": write_lp_seconds + solve_seconds + load_seconds,
        "threads_config": SOLVER_THREADS_LABEL,
        "threads_effective": threads,
    }


result_appsi = benchmark_appsi_highs(
    instance, LP_PATH, threads=SOLVER_THREADS_EFFECTIVE,
)
print("appsi_highs:", result_appsi)
if result_appsi.get("lp_size_mb"):
    print(f"LP escrito en celda 3: {LP_PATH} ({result_appsi['lp_size_mb']:.2f} MB)")

appsi_highs: {'backend': 'appsi_highs', 'status': 'unknown', 'objective': 0.0, 'write_lp_seconds': 8.404292508999788, 'lp_size_mb': 110.74879837036133, 'solve_seconds': 40.204091255000094, 'load_solution_seconds': 0.0, 'total_seconds': 48.60838376399988, 'threads_config': 'all(16)', 'threads_effective': 16}
LP escrito en celda 3: /tmp/osemosys_benchmark_0id0kjvz/model.lp (110.75 MB)


In [18]:
# Celda 4 — Benchmark 2: write LP + highspy directo + mapeo a Pyomo


def pyomo_name_to_lp(name: str) -> str:
    if "[" in name and name.endswith("]"):
        base, rest = name.split("[", 1)
        return f"{base}({rest[:-1]})"
    return name


def lp_name_to_pyomo(name: str) -> str:
    if "(" in name and name.endswith(")"):
        base, rest = name.split("(", 1)
        return f"{base}[{rest[:-1]}]"
    return name


def highs_status_label(status: object) -> str:
    mapping = {
        getattr(highspy.HighsModelStatus, "kOptimal", None): "optimal",
        getattr(highspy.HighsModelStatus, "kInfeasible", None): "infeasible",
        getattr(highspy.HighsModelStatus, "kUnbounded", None): "unbounded",
    }
    for hs, label in mapping.items():
        if hs is not None and status == hs:
            return label
    return str(status)


def apply_highspy_solution(instance, h: highspy.Highs) -> float:
    solution = h.getSolution()
    lp = h.getLp()
    col_names = list(getattr(lp, "col_names_", []) or [])
    col_values = list(getattr(solution, "col_value", []) or [])

    col_map: dict[str, float] = {}
    for idx, name in enumerate(col_names):
        if idx < len(col_values):
            col_map[name] = float(col_values[idx])
            col_map[lp_name_to_pyomo(name)] = float(col_values[idx])

    for var in instance.component_data_objects(Var, active=True):
        pyomo_name = var.name
        lp_name = pyomo_name_to_lp(pyomo_name)
        val = col_map.get(pyomo_name)
        if val is None:
            val = col_map.get(lp_name)
        if val is not None:
            var.set_value(val, skip_validation=True)

    try:
        info = h.getInfo()
        return float(getattr(info, "objective_function_value", 0.0))
    except Exception:
        return float(pyo.value(instance.OBJ))


def benchmark_highspy_via_lp(
    instance,
    lp_path: Path,
    *,
    solver_method: str = "ipm",
    presolve: str = "on",
    parallel: str = "on",
    threads: int = 0,
    skip_write_lp: bool = False,
) -> dict:
    lp_path = Path(lp_path)
    lp_path.parent.mkdir(parents=True, exist_ok=True)

    if skip_write_lp and lp_path.is_file():
        write_lp_seconds = 0.0
        lp_size_mb = lp_path.stat().st_size / (1024 * 1024)
    else:
        t_write = perf_counter()
        instance.write(
            filename=str(lp_path),
            io_options={"symbolic_solver_labels": True},
        )
        write_lp_seconds = perf_counter() - t_write
        lp_size_mb = lp_path.stat().st_size / (1024 * 1024)

    h = highspy.Highs()
    h.setOptionValue("log_to_console", False)
    h.setOptionValue("output_flag", False)
    h.setOptionValue("solver", solver_method)
    h.setOptionValue("presolve", presolve)
    h.setOptionValue("parallel", parallel)
    h.setOptionValue("threads", threads)

    t_read = perf_counter()
    h.readModel(str(lp_path))
    read_model_seconds = perf_counter() - t_read

    t_run = perf_counter()
    h.run()
    run_seconds = perf_counter() - t_run

    status = highs_status_label(h.getModelStatus())
    obj = 0.0
    map_seconds = 0.0
    if "optimal" in status.lower():
        t_map = perf_counter()
        obj = apply_highspy_solution(instance, h)
        map_seconds = perf_counter() - t_map

    return {
        "backend": "highspy_lp",
        "status": status,
        "objective": obj,
        "write_lp_seconds": write_lp_seconds,
        "read_model_seconds": read_model_seconds,
        "run_seconds": run_seconds,
        "map_solution_seconds": map_seconds,
        "total_seconds": write_lp_seconds + read_model_seconds + run_seconds + map_seconds,
        "lp_size_mb": lp_size_mb,
        "solver_method": solver_method,
        "presolve": presolve,
        "parallel": parallel,
        "threads_config": SOLVER_THREADS_LABEL,
        "threads_effective": threads,
    }


result_highspy = benchmark_highspy_via_lp(
    instance,
    LP_PATH,
    solver_method="ipm",
    presolve="on",
    parallel="on",
    threads=SOLVER_THREADS_EFFECTIVE,
    skip_write_lp=True,
)
print("highspy via LP:", result_highspy)
print(f"LP reutilizado: {LP_PATH} ({result_highspy['lp_size_mb']:.2f} MB)")

highspy via LP: {'backend': 'highspy_lp', 'status': 'HighsModelStatus.kNotset', 'objective': 0.0, 'write_lp_seconds': 0.0, 'read_model_seconds': 2.7243152880000707, 'run_seconds': 0.002252880999549234, 'map_solution_seconds': 0.0, 'total_seconds': 2.72656816899962, 'lp_size_mb': 110.74879837036133, 'solver_method': 'ipm', 'presolve': 'on', 'parallel': 'on', 'threads_config': 'all(16)', 'threads_effective': 16}
LP reutilizado: /tmp/osemosys_benchmark_0id0kjvz/model.lp (110.75 MB)


In [19]:
# Celda 5 — Variantes de configuración HiGHS (solo fase solve: readModel + run)
# Reutiliza el .lp ya escrito para no penalizar write_lp en cada variante.
# Con modelo regional (muchas tecnologías) cada readModel+run puede tardar varios minutos.

SKIP_VARIANTS = True  # False = ejecutar ipm/simplex (puede tardar >30 min cada uno)

VARIANTS = [
    {"solver_method": "ipm", "presolve": "on", "parallel": "on"},
    {"solver_method": "simplex", "presolve": "on", "parallel": "on"},
]

variant_rows = []
if SKIP_VARIANTS:
    print("SKIP_VARIANTS=True — variantes omitidas.")
elif not LP_PATH.is_file():
    raise FileNotFoundError("Ejecuta la celda 4 primero para generar el .lp")
else:
    for cfg in VARIANTS:
        h = highspy.Highs()
        h.setOptionValue("log_to_console", False)
        h.setOptionValue("output_flag", False)
        h.setOptionValue("solver", cfg["solver_method"])
        h.setOptionValue("presolve", cfg["presolve"])
        h.setOptionValue("parallel", cfg["parallel"])
        h.setOptionValue("threads", SOLVER_THREADS_EFFECTIVE)

        t0 = perf_counter()
        h.readModel(str(LP_PATH))
        read_s = perf_counter() - t0
        t1 = perf_counter()
        h.run()
        run_s = perf_counter() - t1
        status = highs_status_label(h.getModelStatus())
        try:
            obj = float(h.getInfo().objective_function_value)
        except Exception:
            obj = float("nan")

        variant_rows.append({
            **cfg,
            "status": status,
            "objective": obj,
            "read_model_seconds": read_s,
            "run_seconds": run_s,
            "total_seconds": read_s + run_s,
        })

df_variants = pd.DataFrame(variant_rows)
if not df_variants.empty:
    display(df_variants)

SKIP_VARIANTS=True — variantes omitidas.


In [20]:
# Celda 6 — Tabla comparativa y conclusiones

print(f"Comparación con hilos: {SOLVER_THREADS_LABEL} (efectivo={SOLVER_THREADS_EFFECTIVE})")

rows = [
    {
        "metodo": "appsi_highs (actual)",
        "hilos": result_appsi.get("threads_config", SOLVER_THREADS_LABEL),
        "status": result_appsi["status"],
        "objective": result_appsi["objective"],
        "solve_run_seconds": result_appsi["solve_seconds"],
        "extra_seconds": (
            result_appsi.get("write_lp_seconds", 0)
            + result_appsi["load_solution_seconds"]
        ),
        "total_seconds": result_appsi["total_seconds"],
    },
    {
        "metodo": "highspy via LP (ipm)",
        "hilos": result_highspy.get("threads_config", SOLVER_THREADS_LABEL),
        "status": result_highspy["status"],
        "objective": result_highspy["objective"],
        "solve_run_seconds": result_highspy["run_seconds"],
        "extra_seconds": (
            result_highspy["write_lp_seconds"]
            + result_highspy["read_model_seconds"]
            + result_highspy["map_solution_seconds"]
        ),
        "total_seconds": result_highspy["total_seconds"],
    },
]
df_compare = pd.DataFrame(rows)
display(df_compare)

obj_a = result_appsi["objective"]
obj_b = result_highspy["objective"]
if obj_a and obj_b:
    rel_diff = abs(obj_a - obj_b) / max(abs(obj_a), 1e-9)
    print(f"Diferencia relativa objetivo: {rel_diff:.6e}")
    if rel_diff < 1e-4:
        print("OK: objetivos coherentes entre ambos métodos.")
    else:
        print("ATENCIÓN: revisar tolerancias o mapeo LP -> Pyomo.")

t_appsi = result_appsi["total_seconds"]
t_lp = result_highspy["total_seconds"]
if t_appsi > 0:
    pct = (t_lp - t_appsi) / t_appsi * 100
    faster = "highspy LP" if t_lp < t_appsi else "appsi_highs"
    print(f"Total: appsi={t_appsi:.2f}s, highspy LP={t_lp:.2f}s ({pct:+.1f}% vs appsi)")
    print(f"Más rápido en tiempo total: {faster}")
    print(
        f"Solve puro HiGHS: appsi incluye traducción en solve_seconds; "
        f"highspy run_seconds={result_highspy['run_seconds']:.2f}s"
    )

Comparación con hilos: all(16) (efectivo=16)


,metodo,hilos,status,objective,solve_run_seconds,extra_seconds,total_seconds
0,appsi_highs (actual),all(16),unknown,0.0,40.204091,8.404293,48.608384
1,highspy via LP (ipm),all(16),HighsModelStatus.kNotset,0.0,0.002253,2.724315,2.726568


Total: appsi=48.61s, highspy LP=2.73s (-94.4% vs appsi)
Más rápido en tiempo total: highspy LP
Solve puro HiGHS: appsi incluye traducción en solve_seconds; highspy run_seconds=0.00s


## Celda 7 — Placeholder: linopy (Alternativa B)

**Estado:** no implementado en este benchmark.

**Motivo:** [linopy](https://linopy.readthedocs.io/) reemplazaría Pyomo definiendo variables y restricciones con xarray/pandas. Eso implica reescribir [`model_definition.py`](../backend/app/simulation/core/model_definition.py) (~1500 líneas de restricciones OSeMOSYS).

**Esfuerzo estimado:** varias semanas + regresión completa vs notebook de referencia.

**Próximo paso sugerido:** si highspy-via-LP gana en *solve* pero Pyomo sigue dominando en *build*, evaluar linopy solo para un submodelo (p. ej. balance energético) como prueba de concepto.

```bash
# Cuando se retome:
# pip install linopy
# https://linopy.readthedocs.io/en/latest/
```

In [21]:
# Opcional: limpiar directorio temporal de trabajo
# shutil.rmtree(WORK_DIR, ignore_errors=True)
# print("WORK_DIR eliminado")